# שלב 3 — שיטה B: Suffix Array + HyperLogLog

הגישה האלגוריתמית הקלאסית לבעיית "תת-מחרוזת נפוצה". כל מסלול = "מחרוזת" של תאי H3. מחברים את כל המסלולים (עם **מפרידים ייחודיים** כדי שתת-מסלול לא יחצה בין נסיעות), בונים **Suffix Array + LCP**, וקוראים מהם את כל תתי-המסלולים החוזרים.

**למה זה יעיל:** מיון יחיד של כל הסיפות נותן את *כל* תתי-המחרוזות החוזרות דרך מערך ה-LCP — במקום מעבר נפרד לכל אורך (כמו בשיטת הגרף).

**HyperLogLog:** לכל תת-מסלול מועמד צריך לספור כמה **נסיעות ייחודיות** עברו בו (תמיכה). HLL נותן ספירת-ייחודיים מקורבת בזיכרון זעיר (קילו-בייטים), במקום `set` מדויק.

**אותה הגדרת מקסימליות כמו שיטה A** — כדי שההשוואה תהיה הוגנת.

> הערה: בניית Suffix Array גלובלי היא בעיקרה חישוב על ה-driver (פחות מבוזר מ-Spark) — זו נקודת השוואה חשובה למצגת.


## 1. SparkSession + קונפיגורציה

In [ ]:
from pyspark.sql import SparkSession, functions as F
import numpy as np, hashlib, time, json
from math import radians, sin, cos, asin, sqrt
import h3

spark = (SparkSession.builder.appName("porto-taxi-stage3-suffix")
         .master("local[*]").config("spark.driver.memory", "4g").getOrCreate())
spark.sparkContext.setLogLevel("WARN")

USE_SAMPLE = True
if USE_SAMPLE:
    INPUT_PATH  = "out/trips_h3_sample.parquet"
    OUTPUT_PATH = "out/subroutes_suffix_sample.parquet"
else:
    INPUT_PATH  = "gs://YOUR_BUCKET/encoded/trips_h3.parquet"
    OUTPUT_PATH = "gs://YOUR_BUCKET/results/subroutes_suffix.parquet"

X_PCT = 0.5
CONFIGS_KM = [1, 3, 5, 10, 20, 40]
HLL_P = 12          # 2^12 = 4096 registers


## 2. Suffix Array (prefix-doubling) + LCP (Kasai)

In [ ]:
def suffix_array(s):
    s = np.asarray(s, dtype=np.int64); n = len(s)
    _, rank = np.unique(s, return_inverse=True); rank = rank.astype(np.int64); k = 1
    while True:
        rank2 = np.full(n, -1, dtype=np.int64); rank2[:n-k] = rank[k:]
        order = np.lexsort((rank2, rank))
        rs, r2s = rank[order], rank2[order]
        diff = np.empty(n, dtype=bool); diff[0] = False
        diff[1:] = (rs[1:] != rs[:-1]) | (r2s[1:] != r2s[:-1])
        newrank = np.empty(n, dtype=np.int64); newrank[order] = np.cumsum(diff); rank = newrank
        if rank.max() == n-1: break
        k *= 2
    return order

def kasai_lcp(s, sa):
    s = np.asarray(s, dtype=np.int64); n = len(s)
    rank = np.empty(n, dtype=np.int64); rank[sa] = np.arange(n)
    lcp = np.zeros(n, dtype=np.int64); h = 0
    for i in range(n):
        if rank[i] > 0:
            j = sa[rank[i]-1]
            while i+h < n and j+h < n and s[i+h] == s[j+h]: h += 1
            lcp[rank[i]] = h
            if h > 0: h -= 1
        else: h = 0
    return lcp


## 3. HyperLogLog (ספירת נסיעות ייחודיות מקורבת)

In [ ]:
class HLL:
    def __init__(self, p=12):
        self.p = p; self.m = 1 << p
        self.reg = np.zeros(self.m, dtype=np.int8)
        self.alpha = 0.7213 / (1 + 1.079 / self.m)
    def add_int(self, x):
        h = int.from_bytes(hashlib.blake2b(int(x).to_bytes(8, "little"), digest_size=8).digest(), "little")
        idx = h & (self.m - 1); w = h >> self.p
        rho = (64 - self.p) - w.bit_length() + 1
        if rho > self.reg[idx]: self.reg[idx] = rho
    def count(self):
        m = self.m
        E = self.alpha * m * m / (1.0 / np.power(2.0, self.reg.astype(np.float64))).sum()
        if E <= 2.5 * m:
            v = int((self.reg == 0).sum())
            if v > 0: E = m * np.log(m / v)
        return int(E)


## 4. בניית הרצף + כריית תתי-מסלולים פופולריים

ממפים תאים ל-int, משרשרים עם מפריד ייחודי לכל נסיעה (כך ש-substring שמכיל מפריד שייך לנסיעה אחת בלבד ולעולם לא יהיה נפוץ), ואז סורקים את ה-LCP לכל אורך.

In [ ]:
def build_sequence(cell_lists):
    vocab = {}
    for cells in cell_lists:
        for c in cells:
            if c not in vocab: vocab[c] = len(vocab)
    C = len(vocab); seq, doc, sep = [], [], len(vocab)
    for t, cells in enumerate(cell_lists):
        for c in cells: seq.append(vocab[c]); doc.append(t)
        seq.append(sep); doc.append(-1); sep += 1
    return np.array(seq), np.array(doc), {v: k for k, v in vocab.items()}, C

def mine_popular(seq, doc, sa, lcp, C, minsup, use_hll=False, hll_p=12):
    n = len(seq); maxL = int(lcp.max()); out = {}; hll_err = []
    for ell in range(2, maxL + 1):
        i = 0
        while i < n:
            j = i
            while j + 1 < n and lcp[j+1] >= ell: j += 1
            if j > i:
                start = sa[i]
                if start + ell <= n:
                    sub = seq[start:start+ell]
                    if sub.max() < C:                       # no sentinel -> within one trip
                        docs = doc[sa[i:j+1]]; docs = np.unique(docs[docs >= 0]); exact = len(docs)
                        if use_hll:
                            h = HLL(hll_p)
                            for d in docs: h.add_int(int(d))
                            est = h.count(); hll_err.append((exact, est)); supp = est
                        else:
                            supp = exact
                        if supp >= minsup: out[tuple(int(x) for x in sub)] = exact
            i = j + 1
    return out, hll_err

def maximal_filter(keys):
    ext = set()
    for k in keys:
        if len(k) >= 3: ext.add(k[:-1]); ext.add(k[1:])
        elif len(k) == 2: ext.add((k[0],)); ext.add((k[1],))
    return [k for k in keys if k not in ext]

def route_km(idt, id2cell):
    pts = [h3.cell_to_latlng(id2cell[i]) for i in idt]; t = 0.0
    for (la1, lo1), (la2, lo2) in zip(pts, pts[1:]):
        dlat, dlon = radians(la2-la1), radians(lo2-lo1)
        a = sin(dlat/2)**2 + cos(radians(la1))*cos(radians(la2))*sin(dlon/2)**2
        t += 2 * 6371 * asin(sqrt(a))
    return float(t)


## 5. הרצה

In [ ]:
_t0 = time.time()
cell_lists = [r["cells"] for r in spark.read.parquet(INPUT_PATH).select("cells").collect()]
N = len(cell_lists); minsup = max(2, int(N * X_PCT / 100))
print(f"trips N={N}  minsup={minsup}")

seq, doc, id2cell, C = build_sequence(cell_lists)
sa = suffix_array(seq); lcp = kasai_lcp(seq, sa)
print(f"seq_len={len(seq):,}  vocab={C}  maxLCP={int(lcp.max())}")

popular, _ = mine_popular(seq, doc, sa, lcp, C, minsup)
maxk = maximal_filter(set(popular.keys()))
print(f"popular substrings={len(popular)}  maximal={len(maxk)}")


## 6. תוצאות לפי קונפיגורציה + שמירה

In [ ]:
import pandas as pd
recs = [(">".join(id2cell[i] for i in k), len(k), popular[k], route_km(k, id2cell)) for k in maxk]
pdf = pd.DataFrame(recs, columns=["win_key", "n_cells", "support", "km"])

for K in CONFIGS_KM:
    print(f"  >= {K:>2} km : {int((pdf.km >= K).sum())} popular sub-routes")
print("\ntop 10 longest:")
print(pdf.sort_values("km", ascending=False).head(10).to_string(index=False))

spark.createDataFrame(pdf).write.mode("overwrite").parquet(OUTPUT_PATH)
print("saved:", OUTPUT_PATH)


## 7. דיוק ה-HyperLogLog מול ספירה מדויקת

In [ ]:
_, hll_err = mine_popular(seq, doc, sa, lcp, C, 2, use_hll=True, hll_p=HLL_P)
rel = [abs(e - x) / x for x, e in hll_err if x > 0]
print(f"HLL p={HLL_P}  registers={1<<HLL_P}  memory={1<<HLL_P} bytes")
print(f"on {len(rel):,} distinct-counts:  mean_rel_err={np.mean(rel)*100:.2f}%  p95_rel_err={np.percentile(rel,95)*100:.2f}%")


### שמירת מטריקות להשוואה

In [ ]:
metrics = {
    "method": "suffix_hll",
    "structure": f"Suffix Array + LCP ; HyperLogLog p={HLL_P}",
    "distributed": False,
    "runtime_sec": round(time.time() - _t0, 1),
    "n_maximal": len(maxk),
    "struct_mem_bytes": int(seq.nbytes + sa.nbytes + lcp.nbytes + doc.nbytes),
    "minsup": minsup, "X_pct": X_PCT,
    "per_config": {str(K): int((pdf.km >= K).sum()) for K in CONFIGS_KM},
}
with open("out/metrics_suffix.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)


## סיכום השיטה

**יתרונות:** מיון יחיד נותן את כל תתי-המחרוזות החוזרות (LCP), מדויק, מהיר מאוד על מכונה אחת. HLL חוסך זיכרון בספירת תמיכה.
**חסרונות:** ה-Suffix Array גלובלי הוא בעיקר חישוב על ה-driver — קשה לבזר ומוגבל בזיכרון מכונה אחת; על הדאטה המלא ייתכן שתצטרכו לדגום/לחלק או driver חזק.

תוצאות זהות לשיטה A (אימות הדדי) — ההשוואה המלאה במחברת `05_comparison.ipynb`.
